In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev_pnl",
    choices=["fq_dev_pnl", "fq_test_pnl", "fq_prod_pnl"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="other",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="pnl_budget_flat_data",
    choices=["brand_ho_allocation_cost","pnl_budget_flat_data"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `fq_dev_extloc_staging`"
).select("url").collect()[0][0]

checkpoint = 'abfss://fq-dev-pnl-container@fqadfstoragedev.dfs.core.windows.net/checkpoints/'

In [0]:
from pyspark.sql.functions import *

cdc_raw_data = spark.read.table(f"{environment}_catalog.bronze.{domain}")
cdc_raw_data.limit(1).display()

In [0]:
%sql
select * from fq_dev_pnl_catalog.silver.pnl_budget_flat_data

In [0]:
%sql CREATE EXTERNAL TABLE IF NOT EXISTS fq_dev_pnl_catalog.silver.pnl_budget_flat_data
USING DELTA
LOCATION 'abfss://fq-dev-pnl-container@fqadfstoragedev.dfs.core.windows.net/silver/pnl_budget_flat_data' 
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
%run "/Workspace/Users/tgh3@foodquest.ae/FoodQuest_PnL.git/FoodQuest Management P&L/Formula & Functions Management P&L"

In [0]:
%sql
use catalog `fq_dev_pnl_catalog`

In [0]:
df = spark.read.table('bronze.pnl_budget_flat_data')

df_coa_master = spark.read.table("bronze.dim_coa_master")
df_coa_master = to_snake_case_df(df_coa_master)
df_location_master = spark.read.table("bronze.dim_location_master")
df_location_master = to_snake_case_df(df_location_master)
df_location_master = df_location_master.withColumn("excel_p&l_name", trim(col("excel_p&l_name")))
df_coa_master_distinct = df_coa_master.dropDuplicates(["mapped_name"]) # Distinct new_grouping
df_coa_master_distinct = df_coa_master_distinct.filter(col('management_details_total') == 'Details') # Details



from pyspark.sql.functions import *
df_filtered = df.filter((col("Store Name").isNotNull()))

# Step 1: Join both master tables upfront
df_all_masters = df_filtered.join(
    df_coa_master_distinct, 
    df_coa_master_distinct["mapped_name"].cast("string") == df_filtered["Column 1"], 
    'inner'
).join(
    df_location_master,
    col("Store name") == df_location_master["excel_p&l_name"],
    'left'
)

df_all_masters = df_all_masters.withColumnsRenamed(
{
    "File name": "month",
    "Year": "year",
    # "Column 1": "account_name",
    "Bud": "amount",
    "Store name": "location"
}
).withColumn(
    "month", 
    upper(substring(col("month"), 1, 3))
)

# # Exclude summary "Pre-Opening Expense" accounts that duplicate totals
# df_all_masters_filtered = df_all_masters.filter(
#     ~col("mapped_name").like('Pre-Opening Expense')
# ).filter(
#     ~col("mapped_name").like('Pre Opening Expenses')
# )

df_final_budget = final_df(df_all_masters)

df_final_budget = df_final_budget.withColumnRenamed('amount', 'budget_amount')

In [0]:
import time
from pyspark.sql.functions import *

(df_final_budget.write
        .option("overwriteSchema", "true")
        .option("delta.columnMapping.mode", "name")
        .mode('overwrite')
        .saveAsTable(f"`{environment}_catalog`.`silver`.`{domain}`", mergeSchema=True)
)



In [0]:
%sql
SELECT 
  COUNT(*) AS total_rows,
  count(distinct year) as distinct_years,
  count(distinct month) as distinct_months,
  count(distinct netsuite_location_name) as distinct_locations
FROM fq_dev_pnl_catalog.silver.pnl_budget_flat_data;

In [0]:
%sql select * from fq_dev_pnl_catalog.silver.pnl_budget_flat_data  where netsuite_location_name is not null;

In [0]:
%sql
select * from fq_dev_pnl_catalog.silver.management_pnl where netsuite_location_name is not null

In [0]:
%sql
SELECT * FROM fq_dev_pnl_catalog.silver.pnl_budget_flat_data

In [0]:
%sql
MERGE INTO fq_dev_pnl_catalog.silver.management_pnl target
            USING (
                SELECT 
                    *
                FROM fq_dev_pnl_catalog.silver.pnl_budget_flat_data
            ) as source
            ON target.year = source.year
                AND target.month = source.month
                AND target.netsuite_location_name = source.netsuite_location_name
                AND target.mapped_name = source.mapped_name
            WHEN MATCHED THEN UPDATE SET
                target.budget_amount = source.budget_amount
            WHEN NOT MATCHED THEN INSERT (
                city,
                management_sort_order,
                location_id,
                store_type,
                major_group,
                detail_total,
                account_name,
                zone,
                sub_group,
                management_details_total,
                type,
                account_type,
                store_open_date2,
                brand_id,
                company_id,
                parent_company,
                country_code,
                group_name,
                management_group,
                year,
                month,
                netsuite_location_name,
                mapped_name,
                budget_amount
            ) VALUES (
                source.city,
                source.management_sort_order,
                source.location_id,
                source.store_type,
                source.major_group,
                source.`Detail/Total`,
                source.account_name,
                source.zone,
                source.sub_group,
                source.management_details_total,
                source.type,
                source.account_type,
                source.store_open_date2,
                source.brand_id,
                source.company_id,
                source.parent_company,
                source.country_code,
                source.`group`,
                source.management_group,
                source.year,
                source.month,
                source.netsuite_location_name,
                source.mapped_name,
                source.budget_amount
            )